This notebook prepares input for "8. Multiome 4 - Fig. 3D-F, S4E-H".

## Setup

In [1]:
renv::load('/oak/stanford/groups/agitler/Shared/Shared_Jupyter_Notebook_Analysis/4.1.1-OG-Multiome/')

library(ArchR)
library(future)
library(dplyr)
library(BSgenome.Mmusculus.UCSC.mm10)
library(pheatmap)
library(ggpubr)
library(ggrepel)
library(ggbreak)
library(cowplot)
library(Seurat)
library(forcats)

if(!dir.exists('/oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome')){
  dir.create('/oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome')
}
setwd('/oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome')

addArchRGenome("mm10")

plan(strategy = "multicore", workers = 16)
options(future.globals.maxSize = 41953040000)

addArchRThreads(threads = 16)


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

## Create control cholinergic neuron ArchR project

In [2]:
#Load ArchR Project
proj_cholinergic_dr <- loadArchRProject('proj_cholinergic_dr')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [3]:
# Keep control nuclei (remove disease nuclei) 
proj_cholinergic_ctl <- proj_cholinergic_dr[proj_cholinergic_dr$Stage %in% "control"]

In [7]:
# Save ArchR project
saveArchRProject(ArchRProj = proj_cholinergic_ctl, outputDirectory = "proj_cholinergic_ctl")

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_cholinergic_ctl

Copying Arrow Files...

Copying Arrow Files (1 of 3)

Copying Arrow Files (2 of 3)

Copying Arrow Files (3 of 3)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 6): Embeddings

Copying Other Files (2 of 6): GroupCoverages

Copying Other Files (3 of 6): LSI_ATAC

Copying Other Files (4 of 6): LSI_RNA

Copying Other Files (5 of 6): PeakCalls

Copying Other Files (6 of 6): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
  

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_cholinergic_ctl 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(23): Sample TSSEnrichment ... cholinergic_type
  cholinergic_type_UMAP_labels
numberOfCells(1): 4307
medianTSS(1): 12.469
medianFrags(1): 10290

## Downsample the data so each cholinergic type has the same number of nuclei
Create proj_cholinergic_ctl_downsamp

In [8]:
# Load ArchR project
proj_cholinergic_ctl <- loadArchRProject('proj_cholinergic_ctl')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [16]:
# Get metadata from the project
metadata_df <- getCellColData(proj_cholinergic_ctl, select = c("Sample", "cholinergic_type")) %>% as.data.frame()

# Summarize the number of nuclei for each sample
summary_df <- metadata_df %>%
  group_by(Sample, cholinergic_type) %>%
  summarise(Num_Nuclei = n())

# Summarize the minimum number of nuclei for each sample
min_nuclei_df <- summary_df %>%
  group_by(Sample) %>%
  summarise(Min_Nuclei = min(Num_Nuclei))

min_nuclei_df

`summarise()` has grouped output by 'Sample'. You can override using the `.groups` argument.


Sample,Min_Nuclei
<chr>,<int>
sample1_12_22,38
sample1_5_25,98
sample2_12_22,88


In [17]:
# Add a metadata column containing the sample and cholinergic type information
proj_cholinergic_ctl$Sample_cholType <- paste(proj_cholinergic_ctl$Sample, proj_cholinergic_ctl$cholinergic_type, sep = "_")

In [45]:
(6*38) + (6*98) + (6*88)

[1] 1344

In [48]:
# Get metadata from the project
metadata_df <- getCellColData(proj_cholinergic_ctl, select = c("Sample", "Sample_cholType")) %>% as.data.frame()
metadata_df$Barcode <- rownames(metadata_df)

# Split metadata into a list of data frames based on Sample_Stage
metadata_list <- split(metadata_df, metadata_df$Sample_cholType)

# Function to sample nuclei for each subset
sample_nuclei <- function(df, min_nuclei) {
  if (nrow(df) <= min_nuclei) {
    return(df)
  } else {
    return(df %>% sample_n(size = min_nuclei))
  }
}

# Applying the function using lapply
subsetted_nuclei_list <- lapply(metadata_list, function(df) {
  sample_nuclei(df, min_nuclei_df$Min_Nuclei[min_nuclei_df$Sample == unique(df$Sample)])
})

# Combine the subsets into a single dataframe
subsetted_nuclei <- do.call(rbind, subsetted_nuclei_list)

# Subset the ArchR project
proj_cholinergic_ctl_downsamp <- proj_cholinergic_ctl[as.character(subsetted_nuclei$Barcode), ]

In [49]:
proj_cholinergic_ctl_downsamp


           ___      .______        ______  __    __  .______      
          /   \     |   _  \      /      ||  |  |  | |   _  \     
         /  ^  \    |  |_)  |    |  ,----'|  |__|  | |  |_)  |    
        /  /_\  \   |      /     |  |     |   __   | |      /     
       /  _____  \  |  |\  \\___ |  `----.|  |  |  | |  |\  \\___.
      /__/     \__\ | _| `._____| \______||__|  |__| | _| `._____|
    



class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_cholinergic_ctl 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(24): Sample TSSEnrichment ...
  cholinergic_type_UMAP_labels Sample_cholType
numberOfCells(1): 1344
medianTSS(1): 13.276
medianFrags(1): 10088.5

In [50]:
# Save ArchR project
saveArchRProject(ArchRProj = proj_cholinergic_ctl_downsamp, outputDirectory = "proj_cholinergic_ctl_downsamp")

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_cholinergic_ctl_downsamp

Copying Arrow Files...

Copying Arrow Files (1 of 3)

Copying Arrow Files (2 of 3)

Copying Arrow Files (3 of 3)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 6): Embeddings

Copying Other Files (2 of 6): GroupCoverages

Copying Other Files (3 of 6): LSI_ATAC

Copying Other Files (4 of 6): LSI_RNA

Copying Other Files (5 of 6): PeakCalls

Copying Other Files (6 of 6): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /          

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_cholinergic_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(24): Sample TSSEnrichment ...
  cholinergic_type_UMAP_labels Sample_cholType
numberOfCells(1): 1344
medianTSS(1): 13.276
medianFrags(1): 10088.5

## Create downsampled ArchR projects for each cholinergic type and call peaks
proj_alpha_ctl_downsamp, proj_gamma_ctl_downsamp, proj_gammaStar_ctl_downsamp, proj_Gad1CholInter_ctl_downsamp, proj_Pitx2CholInter_ctl_downsamp, proj_visc_ctl_downsamp

In [2]:
# Load ArchR project
proj_cholinergic_ctl_downsamp <- loadArchRProject('proj_cholinergic_ctl_downsamp')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [4]:
# Split the into ArchR projects for each cholinergic type
proj_alpha_ctl_downsamp <- proj_cholinergic_ctl_downsamp[proj_cholinergic_ctl_downsamp$cholinergic_type == 'Alpha MNs']
proj_gamma_ctl_downsamp <- proj_cholinergic_ctl_downsamp[proj_cholinergic_ctl_downsamp$cholinergic_type == 'Gamma MNs']
proj_gammaStar_ctl_downsamp <- proj_cholinergic_ctl_downsamp[proj_cholinergic_ctl_downsamp$cholinergic_type == 'Gamma* MNs']
proj_Gad1CholInter_ctl_downsamp <- proj_cholinergic_ctl_downsamp[proj_cholinergic_ctl_downsamp$cholinergic_type == 'Gad1+ Cholinergic Interneurons']
proj_Pitx2CholInter_ctl_downsamp <- proj_cholinergic_ctl_downsamp[proj_cholinergic_ctl_downsamp$cholinergic_type == 'Pitx2+ Cholinergic Interneurons']
proj_visc_ctl_downsamp <- proj_cholinergic_ctl_downsamp[proj_cholinergic_ctl_downsamp$cholinergic_type == 'Visceral MNs']

In [5]:
# Save ArchR projects
saveArchRProject(ArchRProj = proj_alpha_ctl_downsamp, outputDirectory = "proj_alpha_ctl_downsamp")
saveArchRProject(ArchRProj = proj_gamma_ctl_downsamp, outputDirectory = "proj_gamma_ctl_downsamp")
saveArchRProject(ArchRProj = proj_gammaStar_ctl_downsamp, outputDirectory = "proj_gammaStar_ctl_downsamp")
saveArchRProject(ArchRProj = proj_Gad1CholInter_ctl_downsamp, outputDirectory = "proj_Gad1CholInter_ctl_downsamp")
saveArchRProject(ArchRProj = proj_Pitx2CholInter_ctl_downsamp, outputDirectory = "proj_Pitx2CholInter_ctl_downsamp")
saveArchRProject(ArchRProj = proj_visc_ctl_downsamp, outputDirectory = "proj_visc_ctl_downsamp")

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_alpha_ctl_downsamp

Copying Arrow Files...

Copying Arrow Files (1 of 3)

Copying Arrow Files (2 of 3)

Copying Arrow Files (3 of 3)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 7): Annotations

Copying Other Files (2 of 7): Embeddings

Copying Other Files (3 of 7): GroupCoverages

Copying Other Files (4 of 7): LSI_ATAC

Copying Other Files (5 of 7): LSI_RNA

Copying Other Files (6 of 7): PeakCalls

Copying Other Files (7 of 7): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
               

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_alpha_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 224
medianTSS(1): 8.7135
medianFrags(1): 23837

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_gamma_ctl_downsamp

Copying Arrow Files...

Copying Arrow Files (1 of 3)

Copying Arrow Files (2 of 3)

Copying Arrow Files (3 of 3)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 7): Annotations

Copying Other Files (2 of 7): Embeddings

Copying Other Files (3 of 7): GroupCoverages

Copying Other Files (4 of 7): LSI_ATAC

Copying Other Files (5 of 7): LSI_RNA

Copying Other Files (6 of 7): PeakCalls

Copying Other Files (7 of 7): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
               

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_gamma_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 224
medianTSS(1): 14.4885
medianFrags(1): 9048.5

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_gammaStar_ctl_downsamp

Copying Arrow Files...

Copying Arrow Files (1 of 3)

Copying Arrow Files (2 of 3)

Copying Arrow Files (3 of 3)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 7): Annotations

Copying Other Files (2 of 7): Embeddings

Copying Other Files (3 of 7): GroupCoverages

Copying Other Files (4 of 7): LSI_ATAC

Copying Other Files (5 of 7): LSI_RNA

Copying Other Files (6 of 7): PeakCalls

Copying Other Files (7 of 7): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
           

...

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_Gad1CholInter_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 224
medianTSS(1): 15.9235
medianFrags(1): 6985

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_Pitx2CholInter_ctl_downsamp

Copying Arrow Files...

Copying Arrow Files (1 of 3)

Copying Arrow Files (2 of 3)

Copying Arrow Files (3 of 3)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 7): Annotations

Copying Other Files (2 of 7): Embeddings

Copying Other Files (3 of 7): GroupCoverages

Copying Other Files (4 of 7): LSI_ATAC

Copying Other Files (5 of 7): LSI_RNA

Copying Other Files (6 of 7): PeakCalls

Copying Other Files (7 of 7): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
      

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_Pitx2CholInter_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 224
medianTSS(1): 13.047
medianFrags(1): 10279

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_visc_ctl_downsamp

Copying Arrow Files...

Copying Arrow Files (1 of 3)

Copying Arrow Files (2 of 3)

Copying Arrow Files (3 of 3)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 7): Annotations

Copying Other Files (2 of 7): Embeddings

Copying Other Files (3 of 7): GroupCoverages

Copying Other Files (4 of 7): LSI_ATAC

Copying Other Files (5 of 7): LSI_RNA

Copying Other Files (6 of 7): PeakCalls

Copying Other Files (7 of 7): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_visc_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 224
medianTSS(1): 13.8125
medianFrags(1): 8458.5

### Call peaks for downsampled cholinergic type ArchR projects

In [6]:
# Load ArchR projects
proj_alpha_ctl_downsamp <- loadArchRProject('proj_alpha_ctl_downsamp')
proj_gamma_ctl_downsamp <- loadArchRProject('proj_gamma_ctl_downsamp')
proj_gammaStar_ctl_downsamp <- loadArchRProject('proj_gammaStar_ctl_downsamp')
proj_Gad1CholInter_ctl_downsamp <- loadArchRProject('proj_Gad1CholInter_ctl_downsamp')
proj_Pitx2CholInter_ctl_downsamp <- loadArchRProject('proj_Pitx2CholInter_ctl_downsamp')
proj_visc_ctl_downsamp <- loadArchRProject('proj_visc_ctl_downsamp')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [7]:
# Create a list containing ArchR projects for each cholinergic type
cholinergic_list <- list(proj_alpha_ctl_downsamp, proj_gamma_ctl_downsamp, proj_gammaStar_ctl_downsamp,
                                 proj_Gad1CholInter_ctl_downsamp, proj_Pitx2CholInter_ctl_downsamp, proj_visc_ctl_downsamp)

In [8]:
table(getCellColData(proj_alpha_ctl_downsamp, "Sample")[,1])


sample1_12_22  sample1_5_25 sample2_12_22 
           38            98            88 

In [9]:
table(getCellColData(proj_gamma_ctl_downsamp, "Sample")[,1])


sample1_12_22  sample1_5_25 sample2_12_22 
           38            98            88 

In [10]:
#Call Peaks w/ Macs2
pathToMacs2 <- findMacs2()

Searching For MACS2..

Found with $path!



In [11]:
# Function to call peaks grouped by Sample
callPeaks <- function(proj) {
    #Make Pseudo-bulk Replicates
    proj <- addGroupCoverages(
        ArchRProj = proj, 
        groupBy = "Sample",
        minCells = 38,
        minReplicates = 3,
        sampleRatio = 0
    )
         
    proj <- addReproduciblePeakSet(
        ArchRProj = proj, 
        groupBy = "Sample", 
        pathToMacs2 = pathToMacs2
    )
    
    proj <- addPeakMatrix(proj)
  
    return(proj)
}

# Iterate over each project in the list and call peaks
cholinergic_ctl_downsamp_list_wpeaks <- lapply(cholinergic_list, callPeaks)

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-634157acbde1-Date-2025-08-05_Time-20-46-17.log
If there is an issue, please report to github with logFile!

sample1_5_25 (1 of 3) : CellGroups N = 3

sample1_12_22 (2 of 3) : CellGroups N = 3

sample2_12_22 (3 of 3) : CellGroups N = 3

2025-08-05 20:46:55 : Creating Coverage Files!, 0.641 mins elapsed.

2025-08-05 20:46:55 : Batch Execution w/ safelapply!, 0.641 mins elapsed.

2025-08-05 20:47:23 : Adding Kmer Bias to Coverage Files!, 1.108 mins elapsed.

Completed Kmer Bias Calculation

Adding Kmer Bias (1 of 9)

Adding Kmer Bias (2 of 9)

Adding Kmer Bias (3 of 9)

Adding Kmer Bias (4 of 9)

Adding Kmer Bias (5 of 9)

Adding Kmer Bias (6 of 9)

Adding Kmer Bias (7 of 9)

Adding Kmer Bias (8 of 9)

Adding Kmer Bias (9 of 9)

2025-08-05 20:47:53 : Finished Creation of Coverage Files!, 1.6 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addGroupCoverages-634157acbde1-Date-2025-08-05_Time-20-46-17.log

ArchR logging to : A

                      Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
sample1_5_25   sample1_5_25     98         67           3   32   33    33500
sample1_12_22 sample1_12_22     38         37           3   18   25    18500
sample2_12_22 sample2_12_22     88         70           3   30   32    35000


2025-08-05 20:47:55 : Batching Peak Calls!, 0.022 mins elapsed.

2025-08-05 20:47:55 : Batch Execution w/ safelapply!, 0 mins elapsed.

2025-08-05 20:48:38 : Identifying Reproducible Peaks!, 0.741 mins elapsed.

2025-08-05 20:48:44 : Creating Union Peak Set!, 0.836 mins elapsed.

Converged after 3 iterations!

Plotting Ggplot!

2025-08-05 20:48:48 : Finished Creating Union Peak Set (58593)!, 0.9 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-634140a055f1-Date-2025-08-05_Time-20-48-48.log
If there is an issue, please report to github with logFile!

2025-08-05 20:48:49 : Batch Execution w/ safelapply!, 0 mins elapsed.

Overriding previous entry for ReadsInPeaks

Overriding previous entry for FRIP

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-634140a055f1-Date-2025-08-05_Time-20-48-48.log

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-6341152a48e1-Date-2025-08-05_Time-20-49-50.log
If there is an issue, please report to github with logFile!

sample1_5

                      Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
sample1_5_25   sample1_5_25     98         67           3   32   33    33500
sample1_12_22 sample1_12_22     38         37           3   18   25    18500
sample2_12_22 sample2_12_22     88         70           3   30   32    35000


2025-08-05 20:51:12 : Batching Peak Calls!, 0.065 mins elapsed.

2025-08-05 20:51:12 : Batch Execution w/ safelapply!, 0 mins elapsed.

2025-08-05 20:51:26 : Identifying Reproducible Peaks!, 0.301 mins elapsed.

2025-08-05 20:51:29 : Creating Union Peak Set!, 0.354 mins elapsed.

Converged after 3 iterations!

Plotting Ggplot!

2025-08-05 20:51:32 : Finished Creating Union Peak Set (42417)!, 0.4 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-63417989030a-Date-2025-08-05_Time-20-51-32.log
If there is an issue, please report to github with logFile!

2025-08-05 20:51:33 : Batch Execution w/ safelapply!, 0 mins elapsed.

Overriding previous entry for ReadsInPeaks

Overriding previous entry for FRIP

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-63417989030a-Date-2025-08-05_Time-20-51-32.log

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-63417fe92bc9-Date-2025-08-05_Time-20-52-32.log
If there is an issue, please report to github with logFile!

sample1_5

...

2025-08-05 20:56:22 : Batching Peak Calls!, 0.038 mins elapsed.

2025-08-05 20:56:22 : Batch Execution w/ safelapply!, 0 mins elapsed.

2025-08-05 20:56:34 : Identifying Reproducible Peaks!, 0.236 mins elapsed.

2025-08-05 21:00:41 : Creating Union Peak Set!, 4.362 mins elapsed.

Converged after 3 iterations!

Plotting Ggplot!

2025-08-05 21:00:44 : Finished Creating Union Peak Set (32797)!, 4.413 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-634127d1612-Date-2025-08-05_Time-21-00-44.log
If there is an issue, please report to github with logFile!

2025-08-05 21:00:45 : Batch Execution w/ safelapply!, 0 mins elapsed.

Overriding previous entry for ReadsInPeaks

Overriding previous entry for FRIP

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-634127d1612-Date-2025-08-05_Time-21-00-44.log

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-63412d919043-Date-2025-08-05_Time-21-01-46.log
If there is an issue, please report to github with logFile!

sample1_5

                      Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
sample1_5_25   sample1_5_25     98         67           3   32   33    33500
sample1_12_22 sample1_12_22     38         37           3   18   25    18500
sample2_12_22 sample2_12_22     88         70           3   30   32    35000


2025-08-05 21:03:04 : Batching Peak Calls!, 0.043 mins elapsed.

2025-08-05 21:03:05 : Batch Execution w/ safelapply!, 0 mins elapsed.

2025-08-05 21:03:24 : Identifying Reproducible Peaks!, 0.376 mins elapsed.

2025-08-05 21:03:28 : Creating Union Peak Set!, 0.434 mins elapsed.

Converged after 3 iterations!

Plotting Ggplot!

2025-08-05 21:03:31 : Finished Creating Union Peak Set (53417)!, 0.485 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-6341371cce20-Date-2025-08-05_Time-21-03-31.log
If there is an issue, please report to github with logFile!

2025-08-05 21:03:33 : Batch Execution w/ safelapply!, 0 mins elapsed.

Overriding previous entry for ReadsInPeaks

Overriding previous entry for FRIP

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-6341371cce20-Date-2025-08-05_Time-21-03-31.log

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-6341512a6052-Date-2025-08-05_Time-21-04-34.log
If there is an issue, please report to github with logFile!

sample1

                      Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
sample1_5_25   sample1_5_25     98         67           3   32   33    33500
sample1_12_22 sample1_12_22     38         37           3   18   25    18500
sample2_12_22 sample2_12_22     88         70           3   30   32    35000


2025-08-05 21:05:47 : Batching Peak Calls!, 0.032 mins elapsed.

2025-08-05 21:05:47 : Batch Execution w/ safelapply!, 0 mins elapsed.

2025-08-05 21:06:01 : Identifying Reproducible Peaks!, 0.269 mins elapsed.

2025-08-05 21:06:04 : Creating Union Peak Set!, 0.323 mins elapsed.

Converged after 3 iterations!

Plotting Ggplot!

2025-08-05 21:06:07 : Finished Creating Union Peak Set (40243)!, 0.372 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-6341ca18593-Date-2025-08-05_Time-21-06-07.log
If there is an issue, please report to github with logFile!

2025-08-05 21:06:08 : Batch Execution w/ safelapply!, 0 mins elapsed.

Overriding previous entry for ReadsInPeaks

Overriding previous entry for FRIP

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-6341ca18593-Date-2025-08-05_Time-21-06-07.log



In [12]:
# Save ArchR projects
saveArchRProject(ArchRProj = cholinergic_ctl_downsamp_list_wpeaks[[1]], outputDirectory = "proj_alpha_ctl_downsamp")
saveArchRProject(ArchRProj = cholinergic_ctl_downsamp_list_wpeaks[[2]], outputDirectory = "proj_gamma_ctl_downsamp")
saveArchRProject(ArchRProj = cholinergic_ctl_downsamp_list_wpeaks[[3]], outputDirectory = "proj_gammaStar_ctl_downsamp")
saveArchRProject(ArchRProj = cholinergic_ctl_downsamp_list_wpeaks[[4]], outputDirectory = "proj_Gad1CholInter_ctl_downsamp")
saveArchRProject(ArchRProj = cholinergic_ctl_downsamp_list_wpeaks[[5]], outputDirectory = "proj_Pitx2CholInter_ctl_downsamp")
saveArchRProject(ArchRProj = cholinergic_ctl_downsamp_list_wpeaks[[6]], outputDirectory = "proj_visc_ctl_downsamp")

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
         

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_alpha_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 224
medianTSS(1): 8.7135
medianFrags(1): 23837

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
         

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_gamma_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 224
medianTSS(1): 14.4885
medianFrags(1): 9048.5

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
         

...

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_Gad1CholInter_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 224
medianTSS(1): 15.9235
medianFrags(1): 6985

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
         

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_Pitx2CholInter_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 224
medianTSS(1): 13.047
medianFrags(1): 10279

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
         

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_visc_ctl_downsamp 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 224
medianTSS(1): 13.8125
medianFrags(1): 8458.5

## Motor neurons: differential accessibility with disease
### Downsample the data so each motor neuron type (alpha, pan-gamma, visceral) has the same number of nuclei
Create marker_lists_mn_downsamp.rds

### Create skeletal MN subtype ArchR projects

In [2]:
#Load ArchR Project
proj_cholinergic_dr <- loadArchRProject('proj_cholinergic_dr')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [3]:
proj_mn <- proj_cholinergic_dr[proj_cholinergic_dr$cholinergic_type %in% c('Alpha MNs', 'Gamma MNs', 'Gamma* MNs', 'Visceral MNs')]

In [4]:
#Add motor neuron type annotations (Alpha, Pan-Gamma, and Visceral MNs)
newLabels <- c('Alpha MNs', 'Pan-Gamma MNs', 'Pan-Gamma MNs', 'Visceral MNs')

oldLabels <- c('Alpha MNs', 'Gamma MNs', 'Gamma* MNs', 'Visceral MNs')

proj_mn$mn_type <- mapLabels(proj_mn$cholinergic_type, newLabels = newLabels, oldLabels = oldLabels)

In [5]:
# Get metadata from the project
metadata_df <- getCellColData(proj_mn, select = c("Sample", "Stage", "mn_type")) %>% as.data.frame()

# Summarize the number of nuclei for each sample
summary_df <- metadata_df %>%
  group_by(Sample, Stage, mn_type) %>%
  summarise(Num_Nuclei = n())

# Summarize the minimum number of nuclei for each sample
min_nuclei_df <- summary_df %>%
  group_by(Sample, Stage) %>%
  summarise(Min_Nuclei = min(Num_Nuclei))

min_nuclei_df$Sample_Stage <- paste(min_nuclei_df$Sample, min_nuclei_df$Stage, sep = "_")
min_nuclei_df

`summarise()` has grouped output by 'Sample', 'Stage'. You can override using the `.groups` argument.
`summarise()` has grouped output by 'Sample'. You can override using the `.groups` argument.


Sample,Stage,Min_Nuclei,Sample_Stage
<chr>,<chr>,<int>,<chr>
sample1_12_22,control,89,sample1_12_22_control
sample1_12_22,early,151,sample1_12_22_early
sample1_5_25,control,294,sample1_5_25_control
sample1_5_25,mid-late,481,sample1_5_25_mid-late
sample1_6_3,mid-late,453,sample1_6_3_mid-late
sample2_12_22,control,191,sample2_12_22_control
sample2_12_22,early,94,sample2_12_22_early
sample2_6_3,mid-late,542,sample2_6_3_mid-late


In [6]:
# Add a metadata column containing the sample and stage information
proj_mn$Sample_Stage <- paste(proj_mn$Sample, proj_mn$Stage, sep = "_")

# Split the into ArchR projects for each motor neuron type
proj_alpha_mn_downsamp <- proj_mn[proj_mn$mn_type == 'Alpha MNs']
proj_panGamma_mn_downsamp <- proj_mn[proj_mn$mn_type == 'Pan-Gamma MNs']
proj_visc_mn_downsamp <- proj_mn[proj_mn$mn_type == 'Visceral MNs']

In [7]:
# Save ArchR projects
saveArchRProject(ArchRProj = proj_alpha_mn_downsamp, outputDirectory = "proj_alpha_mn_downsamp")
saveArchRProject(ArchRProj = proj_panGamma_mn_downsamp, outputDirectory = "proj_panGamma_mn_downsamp")
saveArchRProject(ArchRProj = proj_visc_mn_downsamp, outputDirectory = "proj_visc_mn_downsamp")

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_alpha_mn_downsamp

Copying Arrow Files...

Copying Arrow Files (1 of 5)

Copying Arrow Files (2 of 5)

Copying Arrow Files (3 of 5)

Copying Arrow Files (4 of 5)

Copying Arrow Files (5 of 5)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 6): Embeddings

Copying Other Files (2 of 6): GroupCoverages

Copying Other Files (3 of 6): LSI_ATAC

Copying Other Files (4 of 6): LSI_RNA

Copying Other Files (5 of 6): PeakCalls

Copying Other Files (6 of 6): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_alpha_mn_downsamp 
samples(5): sample2_6_3 sample1_5_25 sample2_12_22 sample1_6_3
  sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(25): Sample TSSEnrichment ... mn_type Sample_Stage
numberOfCells(1): 3433
medianTSS(1): 8.747
medianFrags(1): 25770

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_panGamma_mn_downsamp

Copying Arrow Files...

Copying Arrow Files (1 of 5)

Copying Arrow Files (2 of 5)

Copying Arrow Files (3 of 5)

Copying Arrow Files (4 of 5)

Copying Arrow Files (5 of 5)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 6): Embeddings

Copying Other Files (2 of 6): GroupCoverages

Copying Other Files (3 of 6): LSI_ATAC

Copying Other Files (4 of 6): LSI_RNA

Copying Other Files (5 of 6): PeakCalls

Copying Other Files (6 of 6): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_panGamma_mn_downsamp 
samples(5): sample2_6_3 sample1_5_25 sample2_12_22 sample1_6_3
  sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(25): Sample TSSEnrichment ... mn_type Sample_Stage
numberOfCells(1): 2344
medianTSS(1): 13.8145
medianFrags(1): 11032.5

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_visc_mn_downsamp

Copying Arrow Files...

Copying Arrow Files (1 of 5)

Copying Arrow Files (2 of 5)

Copying Arrow Files (3 of 5)

Copying Arrow Files (4 of 5)

Copying Arrow Files (5 of 5)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 6): Embeddings

Copying Other Files (2 of 6): GroupCoverages

Copying Other Files (3 of 6): LSI_ATAC

Copying Other Files (4 of 6): LSI_RNA

Copying Other Files (5 of 6): PeakCalls

Copying Other Files (6 of 6): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.


class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_visc_mn_downsamp 
samples(5): sample2_6_3 sample1_5_25 sample2_12_22 sample1_6_3
  sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(25): Sample TSSEnrichment ... mn_type Sample_Stage
numberOfCells(1): 6349
medianTSS(1): 14.312
medianFrags(1): 9783

### Downsample the nuclei

In [8]:
# Load ArchR projects
proj_alpha_mn_downsamp <- loadArchRProject('proj_alpha_mn_downsamp')
proj_panGamma_mn_downsamp <- loadArchRProject('proj_panGamma_mn_downsamp')
proj_visc_mn_downsamp <- loadArchRProject('proj_visc_mn_downsamp')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [9]:
# Create a list containing ArchR projects for each motor neuron type
mn_list <- list(proj_alpha_mn_downsamp, proj_panGamma_mn_downsamp, proj_visc_mn_downsamp)

In [10]:
# Function to downsample cells in an ArchR project
downsample_cells <- function(project, min_nuclei_df) {
  # Get metadata from the project
  metadata_df <- getCellColData(project, select = c("Sample_Stage")) %>% as.data.frame()
  metadata_df$Barcode <- rownames(metadata_df)
  
  # Split metadata into a list of data frames based on Sample_Stage
  metadata_list <- split(metadata_df, metadata_df$Sample_Stage)
  
  # Function to sample nuclei for each subset
  sample_nuclei <- function(df, min_nuclei) {
    if (nrow(df) <= min_nuclei) {
      return(df)
    } else {
      return(df %>% sample_n(size = min_nuclei))
    }
  }
  
  # Apply the function to each subset in the list
  subsetted_nuclei_list <- lapply(names(metadata_list), function(stage) {
    sample_nuclei(metadata_list[[stage]], min_nuclei_df$Min_Nuclei[min_nuclei_df$Sample_Stage == stage])
  })
  
  # Combine the subsets into a single dataframe
  subsetted_nuclei <- do.call(rbind, subsetted_nuclei_list)
  
  # Subset the ArchR project
  project <- project[as.character(subsetted_nuclei$Barcode), ]
  
  return(project)
}

# Iterate over each project in the list and downsample cells
mn_downsamp_list <- lapply(mn_list, downsample_cells, min_nuclei_df)

In [11]:
mn_downsamp_list


           ___      .______        ______  __    __  .______      
          /   \     |   _  \      /      ||  |  |  | |   _  \     
         /  ^  \    |  |_)  |    |  ,----'|  |__|  | |  |_)  |    
        /  /_\  \   |      /     |  |     |   __   | |      /     
       /  _____  \  |  |\  \\___ |  `----.|  |  |  | |  |\  \\___.
      /__/     \__\ | _| `._____| \______||__|  |__| | _| `._____|
    


           ___      .______        ______  __    __  .______      
          /   \     |   _  \      /      ||  |  |  | |   _  \     
         /  ^  \    |  |_)  |    |  ,----'|  |__|  | |  |_)  |    
        /  /_\  \   |      /     |  |     |   __   | |      /     
       /  _____  \  |  |\  \\___ |  `----.|  |  |  | |  |\  \\___.
      /__/     \__\ | _| `._____| \______||__|  |__| | _| `._____|
    


           ___      .______        ______  __    __  .______      
          /   \     |   _  \      /      ||  |  |  | |   _  \     
         /  ^  \    |  |_)  |    |  ,----'|  |_

[[1]]
class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_alpha_mn_downsamp 
samples(5): sample2_6_3 sample1_5_25 sample2_12_22 sample1_6_3
  sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(25): Sample TSSEnrichment ... mn_type Sample_Stage
numberOfCells(1): 2295
medianTSS(1): 8.85
medianFrags(1): 25717

[[2]]
class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_panGamma_mn_downsamp 
samples(5): sample2_6_3 sample1_5_25 sample2_12_22 sample1_6_3
  sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(25): Sample TSSEnrichment ... mn_type Sample_Stage
numberOfCells(1): 2295
medianTSS(1): 13.794
medianFrags(1): 10962

[[3]]
class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_visc_mn_downsamp 
samples(5): sample2_6_3 sample1_5_25 sample2_12_22 sample1_6_3
  sample1_12_22
sampleColData names(1): ArrowFiles
cellColData na

### Call peaks for each motor neuron type (control vs. mid/end)

In [12]:
#Call Peaks w/ Macs2
pathToMacs2 <- findMacs2()

Searching For MACS2..

Found with $path!



In [13]:
# Function to call peaks grouped by disease stage
callDiseasePeaks <- function(proj) {
    #Make Pseudo-bulk Replicates
    proj <- addGroupCoverages(
        ArchRProj = proj, 
        groupBy = "Stage",
        minCells = 89,
        minReplicates = 2,
        sampleRatio = 0
    )
         
    proj <- addReproduciblePeakSet(
        ArchRProj = proj, 
        groupBy = "Stage", 
        pathToMacs2 = pathToMacs2
    )
    
    proj <- addPeakMatrix(proj)
  
    return(proj)
}

# Iterate over each project in the list and call peaks
mn_downsamp_list_wpeaks <- lapply(mn_downsamp_list, callDiseasePeaks)

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-1d9c60d3c534-Date-2024-07-29_Time-11-53-03.log
If there is an issue, please report to github with logFile!

control (1 of 3) : CellGroups N = 3

early (2 of 3) : CellGroups N = 2

mid-late (3 of 3) : CellGroups N = 3

2024-07-29 11:53:26 : Creating Coverage Files!, 0.391 mins elapsed.

2024-07-29 11:53:26 : Batch Execution w/ safelapply!, 0.391 mins elapsed.

2024-07-29 11:54:04 : Adding Kmer Bias to Coverage Files!, 1.011 mins elapsed.

Completed Kmer Bias Calculation

Adding Kmer Bias (1 of 8)

Adding Kmer Bias (2 of 8)

Adding Kmer Bias (3 of 8)

Adding Kmer Bias (4 of 8)

Adding Kmer Bias (5 of 8)

Adding Kmer Bias (6 of 8)

Adding Kmer Bias (7 of 8)

Adding Kmer Bias (8 of 8)

2024-07-29 11:54:37 : Finished Creation of Coverage Files!, 1.562 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addGroupCoverages-1d9c60d3c534-Date-2024-07-29_Time-11-53-03.log

ArchR logging to : ArchRLogs/ArchR-addReproduciblePeakSet-1d9c4

            Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
control   control    574        574           3   89  294   150000
early       early    245        245           2   94  151   122500
mid-late mid-late   1476       1434           3  453  500   150000


2024-07-29 11:54:37 : Batching Peak Calls!, 0.006 mins elapsed.

2024-07-29 11:54:38 : Batch Execution w/ safelapply!, 0 mins elapsed.

2024-07-29 11:58:05 : Identifying Reproducible Peaks!, 3.465 mins elapsed.

2024-07-29 11:58:15 : Creating Union Peak Set!, 3.638 mins elapsed.

Converged after 4 iterations!

Plotting Ggplot!

2024-07-29 11:58:22 : Finished Creating Union Peak Set (174691)!, 3.742 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-1d9c7329b0ba-Date-2024-07-29_Time-11-58-22.log
If there is an issue, please report to github with logFile!

2024-07-29 11:58:22 : Batch Execution w/ safelapply!, 0 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-1d9c7329b0ba-Date-2024-07-29_Time-11-58-22.log

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-1d9c1065ea63-Date-2024-07-29_Time-11-59-25.log
If there is an issue, please report to github with logFile!

control (1 of 3) : CellGroups N = 3

early (2 of 3) : CellGroups N = 2

mid-late (3 of

            Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
control   control    574        574           3   89  294   150000
early       early    245        245           2   94  151   122500
mid-late mid-late   1476       1434           3  453  500   150000


2024-07-29 12:00:34 : Batching Peak Calls!, 0.006 mins elapsed.

2024-07-29 12:00:34 : Batch Execution w/ safelapply!, 0 mins elapsed.

2024-07-29 12:02:19 : Identifying Reproducible Peaks!, 1.749 mins elapsed.

2024-07-29 12:02:25 : Creating Union Peak Set!, 1.862 mins elapsed.

Converged after 4 iterations!

Plotting Ggplot!

2024-07-29 12:02:29 : Finished Creating Union Peak Set (125198)!, 1.92 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-1d9c15a3345-Date-2024-07-29_Time-12-02-29.log
If there is an issue, please report to github with logFile!

2024-07-29 12:02:29 : Batch Execution w/ safelapply!, 0 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-1d9c15a3345-Date-2024-07-29_Time-12-02-29.log

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-1d9c487a539-Date-2024-07-29_Time-12-03-31.log
If there is an issue, please report to github with logFile!

control (1 of 3) : CellGroups N = 3

early (2 of 3) : CellGroups N = 2

mid-late (3 of 3) 

            Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
control   control    574        574           3   89  294   150000
early       early    245        245           2   94  151   122500
mid-late mid-late   1476       1434           3  453  500   150000


2024-07-29 12:04:38 : Batching Peak Calls!, 0.006 mins elapsed.

2024-07-29 12:04:38 : Batch Execution w/ safelapply!, 0 mins elapsed.

2024-07-29 12:06:19 : Identifying Reproducible Peaks!, 1.695 mins elapsed.

2024-07-29 12:06:27 : Creating Union Peak Set!, 1.813 mins elapsed.

Converged after 4 iterations!

Plotting Ggplot!

2024-07-29 12:06:33 : Finished Creating Union Peak Set (136449)!, 1.92 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-1d9c4d6bedcd-Date-2024-07-29_Time-12-06-33.log
If there is an issue, please report to github with logFile!

2024-07-29 12:06:33 : Batch Execution w/ safelapply!, 0 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-1d9c4d6bedcd-Date-2024-07-29_Time-12-06-33.log



In [14]:
# Save ArchR projects
saveArchRProject(ArchRProj = mn_downsamp_list_wpeaks[[1]], outputDirectory = "proj_alpha_mn_downsamp")
saveArchRProject(ArchRProj = mn_downsamp_list_wpeaks[[2]], outputDirectory = "proj_panGamma_mn_downsamp")
saveArchRProject(ArchRProj = mn_downsamp_list_wpeaks[[3]], outputDirectory = "proj_visc_mn_downsamp")

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
         

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_alpha_mn_downsamp 
samples(5): sample2_6_3 sample1_5_25 sample2_12_22 sample1_6_3
  sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(27): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 2295
medianTSS(1): 8.85
medianFrags(1): 25717

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
         

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_panGamma_mn_downsamp 
samples(5): sample2_6_3 sample1_5_25 sample2_12_22 sample1_6_3
  sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(27): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 2295
medianTSS(1): 13.794
medianFrags(1): 10962

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_visceral_mn_downsamp

Copying Arrow Files...

Copying Arrow Files (1 of 5)

Copying Arrow Files (2 of 5)

Copying Arrow Files (3 of 5)

Copying Arrow Files (4 of 5)

Copying Arrow Files (5 of 5)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 6): Embeddings

Copying Other Files (2 of 6): GroupCoverages

Copying Other Files (3 of 6): LSI_ATAC

Copying Other Files (4 of 6): LSI_RNA

Copying Other Files (5 of 6): PeakCalls

Copying Other Files (6 of 6): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_visceral_mn_downsamp 
samples(5): sample2_6_3 sample1_5_25 sample2_12_22 sample1_6_3
  sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(27): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 2295
medianTSS(1): 14.286
medianFrags(1): 9702

### Get differentially accessible peaks for each MN subtype with disease 

In [15]:
# Load ArchR projects
proj_alpha_mn_downsamp <- loadArchRProject('proj_alpha_mn_downsamp')
proj_panGamma_mn_downsamp <- loadArchRProject('proj_panGamma_mn_downsamp')
proj_visc_mn_downsamp <- loadArchRProject('proj_visc_mn_downsamp')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [16]:
# Function to get marker peaks and markers for an ArchR project
get_markers_for_project <- function(project) {
  # Get marker peaks
  markerPeaks <- getMarkerFeatures(
    ArchRProj = project, 
    useMatrix = "PeakMatrix",
    groupBy = "Stage",
    testMethod = "wilcoxon",
    maxCells = 574,
    bias = c("TSSEnrichment", "log10(nFrags)"),
    useGroups = "mid-late", 
    bgdGroups = "control"
  )

  # Get markers
  markerList_up <- getMarkers(markerPeaks, cutOff = "Log2FC > 0", returnGR = TRUE)
  markerList_down <- getMarkers(markerPeaks, cutOff = "Log2FC < 0", returnGR = TRUE)

  markers_up <- markerList_up$'mid-late'
  markers_down <- markerList_down$'mid-late'

  return(list(markers_up = markers_up, markers_down = markers_down))
}

# Apply the function to each ArchR project in cholinergic_list
marker_lists <- lapply(mn_downsamp_list_wpeaks, get_markers_for_project)

ArchR logging to : ArchRLogs/ArchR-getMarkerFeatures-1d9c654d40ad-Date-2024-07-29_Time-12-08-44.log
If there is an issue, please report to github with logFile!

MatrixClass = Sparse.Integer.Matrix

2024-07-29 12:08:45 : Matching Known Biases, 0.006 mins elapsed.

2024-07-29 12:08:48 : Computing Pairwise Tests (1 of 1), 0.059 mins elapsed.

Pairwise Test mid-late : Seqnames chr1

Pairwise Test mid-late : Seqnames chr10

Pairwise Test mid-late : Seqnames chr11

Pairwise Test mid-late : Seqnames chr12

Pairwise Test mid-late : Seqnames chr13

Pairwise Test mid-late : Seqnames chr14

Pairwise Test mid-late : Seqnames chr15

Pairwise Test mid-late : Seqnames chr16

Pairwise Test mid-late : Seqnames chr17

Pairwise Test mid-late : Seqnames chr18

Pairwise Test mid-late : Seqnames chr19

Pairwise Test mid-late : Seqnames chr2

Pairwise Test mid-late : Seqnames chr3

Pairwise Test mid-late : Seqnames chr4

Pairwise Test mid-late : Seqnames chr5

Pairwise Test mid-late : Seqnames chr6

Pairwise

In [17]:
# Save the list to an rds file
saveRDS(marker_lists, file = "marker_lists_mn_downsamp.rds")